# **EDA: ANÁLISIS EXPLORATORIO DE DATOS**

Este notebook contiene la revisión exploratoria inicial de los datasets originales, con el objetivo de evaluar su estructura, calidad y estado general antes de aplicar cualquier proceso de limpieza o transformación.

La revisión se realiza mediante una función estructurada que centraliza la validación del dataset y permite identificar posibles problemas de calidad de datos, inconsistencias y características relevantes para su posterior preparación.

In [8]:
# ============================================================================
# CONFIGURACIÓN DEL ENTORNO
# Importación de librerías para la manipulación, análisis y visualización
# de datos durante la fase exploratoria (EDA).
# ============================================================================

# Manipulación y análisis de datos
import pandas as pd
import numpy as np

# Visualización de datos
import seaborn as sns
import matplotlib.pyplot as plt


# ============================================================================
# CONFIGURACIÓN DE VISUALIZACIÓN EN PANDAS
# Ajuste de opciones para mejorar la legibilidad durante la inspección
# exploratoria, mostrando todas las columnas y un número acotado de filas.
# ============================================================================

pd.set_option("display.max_columns", None)  # mostrar todas las columnas
pd.set_option("display.max_rows", 100)      # limitar la visualización de filas
pd.set_option("display.width", None)        # ajustar automáticamente el ancho



df1 = pd.read_csv("../Data/sin procesar/Customer online delivery dataset - Customer_data.csv", index_col=0)
df2 = pd.read_csv("../Data/sin procesar/food_delivery_dataset.csv",index_col=0)


# ============================================================================
# INSPECCIÓN INICIAL
# Visualización de las primeras filas para verificar la correcta carga,
# estructura general y nombres de columnas.
# ============================================================================

display(df1.head())
display(df2.head())


,Gender,Marital Status,Occupation,Educational Qualifications,Family size,Frequently used Medium,Frequently ordered Meal category,Perference,Restaurnat Rating,Delivery Rating,No. of orders placed,Delivery Time,Order Value,Ease and convenient,Self Cooking,Health Concern,Late Delivery,Poor Hygiene,Bad past experience,More Offers and Discount,Maximum wait time,Influence of rating
Age,,,,,,,,,,,,,,,,,,,,,,
20,Female,Single,Student,Post Graduate,4,Food delivery apps,Breakfast,Non Veg foods (Lunch / Dinner),1,1,150,45,1,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,30 minutes,Yes
24,Female,Single,Student,Graduate,3,Food delivery apps,Snacks,Non Veg foods (Lunch / Dinner),1,1,32,61,3,Strongly agree,Strongly agree,Strongly agree,Agree,Strongly agree,Strongly agree,Strongly agree,30 minutes,Yes
22,Male,Single,Student,Post Graduate,3,Food delivery apps,Lunch,Non Veg foods (Lunch / Dinner),3,5,193,19,1,Strongly agree,Disagree,Neutral,Neutral,Agree,Agree,Neutral,45 minutes,Yes
22,Female,Single,Student,Graduate,6,Food delivery apps,Snacks,Veg foods (Breakfast / Lunch / Dinner),2,4,60,44,1,Agree,Agree,Strongly agree,Neutral,Agree,Disagree,Strongly agree,30 minutes,Yes
22,Male,Single,Student,Post Graduate,4,Walk-in,Lunch,Non Veg foods (Lunch / Dinner),3,1,234,24,3,Agree,Agree,Strongly agree,Strongly agree,Agree,Strongly agree,Agree,30 minutes,Yes


,restaurant_id,food_item,order_time,delivery_time,delivery_distance,order_value,delivery_method,traffic_condition,weather_condition,delivery_delay,route_taken,customer_id,age,gender,location,order_history,customer_rating,preferred_cuisine,order_frequency,loyalty_program,food_temperature,food_freshness,packaging_quality,food_condition,customer_satisfaction,small_route,bike_friendly_route,route_type,route_efficiency,traffic_avoidance
order_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ORD000001,16,Taccos,2024-01-31,2024-01-31,2.17,42.21,Bike,Medium,Snowy,17.15,Route_4,CUST000001,32,Female,Hyderabad,46,4,American,Monthly,Yes,Hot,5,1,Fair,3,False,False,Bike-friendly,0.801291,Yes
ORD000002,30,Briyani rice,2024-10-16,2024-10-16,13.40,24.82,Car,High,Sunny,16.04,Route_2,CUST000002,59,Male,Lucknow,19,1,Asian,Weekly,No,Warm,3,2,Fair,5,False,False,Bike-friendly,0.645795,No
ORD000003,3,Pasta,2024-09-11,2024-09-11,10.74,37.25,Walk,High,Snowy,9.09,Route_3,CUST000003,20,Male,Chennai,10,5,Mexican,Monthly,Yes,Warm,4,5,Good,5,True,False,Bike-friendly,0.291193,No
ORD000004,93,Whole cake,2024-09-17,2024-09-17,6.29,49.88,Bike,High,Snowy,-2.11,Route_5,CUST000004,31,Male,Bangalore,6,4,American,Monthly,Yes,Cold,2,3,Poor,4,False,False,Car-only,0.133827,No
ORD000005,15,Sushi,2024-08-26,2024-08-26,2.94,8.53,Walk,Low,Sunny,12.20,Route_5,CUST000005,27,Other,Ahmedabad,24,3,Asian,Weekly,Yes,Cold,4,5,Good,1,False,True,Car-only,0.349233,No


In [3]:
def eda(df):
    """
    Realiza un análisis exploratorio estructurado (EDA) sobre un DataFrame.

    Parámetros
    ----------
    df : pandas.DataFrame
        Dataset a analizar.

    Devuelve
    --------
    None
        La función no devuelve ningún objeto. Muestra en pantalla el análisis
        exploratorio, estadísticas descriptivas, visualizaciones y flags de calidad.

    Qué analiza
    -----------
    1) Visión general:
       - Vista inicial (head), dimensiones, info() y duplicados.

    2) Calidad de columnas:
       - Tabla resumen por columna (tipos, nulos, cardinalidad).
       - Flags por umbral de nulos (bajo/moderado/alto).

    3) Análisis de nulos:
       - Categóricas: distribución (porcentaje + recuento) incluyendo NaN.
       - Numéricas: % nulos + histograma + KDE (si aplica) + media/mediana/moda.

    4) Estadística descriptiva y controles básicos:
       - Describe numéricas y categóricas.
       - Flags: valores negativos en numéricas; varianza 0 en numéricas.

    5) Outliers:
       - Detección por IQR (1.5x) + boxplot para columnas con outliers.

    Objetivo
    --------
    Facilitar decisiones de limpieza (imputación, tipado, tratamiento de outliers)
    y comprensión del dataset antes de aplicar transformaciones.
    """

    # ==================================================
    # 1) VISIÓN GENERAL
    # ==================================================
    # Objetivo: obtener una primera fotografía del dataset (tamaño, estructura,
    # tipos básicos y presencia de duplicados) antes de entrar en métricas y flags.
    print("📐 VISIÓN GENERAL DEL DATASET")
    print("Vista inicial (primeras 5 filas):")
    display(df.head())

    filas, columnas = df.shape
    print(f"Dimensiones: {filas} filas × {columnas} columnas")

    print("\nInformación general (dtypes, nulos, memoria):")
    df.info()

    duplicados = df.duplicated().sum()
    pct_dup = (duplicados / filas * 100) if filas else 0
    print(f"\n🚩 FLAG: duplicados - Filas duplicadas: {duplicados} ({pct_dup:.2f}%)")
    print("-" * 70)

    # ==================================================
    # 2) CALIDAD DE COLUMNAS
    # ==================================================
    # Objetivo: construir una tabla resumen por columna para identificar
    # rápidamente variables problemáticas (nulos, tipos).
    print("🚑 CALIDAD DE COLUMNAS")

    calidad = (
        pd.DataFrame({
            "Tipo": df.dtypes,
            "Nulos": df.isnull().sum(),
            "% Nulos": (df.isnull().mean() * 100).round(2),
            "Valores Únicos": df.nunique(),
            "% Cardinalidad": (df.nunique()/filas*100).round(2)
            })
        .sort_values("% Nulos", ascending=False)
    )

    display(calidad)

    # ---- Flags por nulos (umbrales) ----
    # Objetivo: priorizar el tratamiento de nulos clasificando columnas según severidad.
    cols_nulos_bajo = calidad[(calidad["% Nulos"] > 0) & (calidad["% Nulos"] <= 5)].index.tolist()
    cols_nulos_moderado = calidad[(calidad["% Nulos"] > 5) & (calidad["% Nulos"] <= 20)].index.tolist()
    cols_nulos_alto = calidad[calidad["% Nulos"] > 20].index.tolist()

    print("\n🚩 FLAG: nulos - Clasificación por umbral (% nulos)")
    print("  🟢 Bajo (≤ 5%):")
    if cols_nulos_bajo:
        for col in cols_nulos_bajo:
            print(f"    - {col} ({calidad.loc[col, '% Nulos']:.2f}%)")
    else:
        print("    - Ninguna")

    print("  🟡 Moderado (5%–20%):")
    if cols_nulos_moderado:
        for col in cols_nulos_moderado:
            print(f"    - {col} ({calidad.loc[col, '% Nulos']:.2f}%)")
    else:
        print("    - Ninguna")

    print("  🔴 Alto (> 20%):")
    if cols_nulos_alto:
        for col in cols_nulos_alto:
            print(f"    - {col} ({calidad.loc[col, '% Nulos']:.2f}%)")
    else:
        print("    - Ninguna")

    # ==================================================
    # 3) ANÁLISIS DE NULOS
    # ==================================================
    # Objetivo: identificar variables con nulos y entender el patrón básico:
    # - En categóricas: distribución de categorías + NaN.
    # - En numéricas: impacto de nulos y forma de la distribución para orientar imputación.

    # --- 3.1 Nulos en categóricas ---
    cols_cat = df.select_dtypes(include=["O"]).columns
    cols_cat_nulos = [c for c in cols_cat if df[c].isnull().any()]

    print("\n🔤 NULOS EN VARIABLES CATEGÓRICAS")
    if cols_cat_nulos:
        for col in cols_cat_nulos:
            pct_nulos = df[col].isnull().mean() * 100
            print(f"\n📌 {col} — 🚩 FLAG: nulos_categorica ({pct_nulos:.2f}%)")

            dist_pct = df[col].value_counts(normalize=True, dropna=False) * 100
            dist_cnt = df[col].value_counts(dropna=False)

            display(pd.DataFrame({
                "Porcentaje": dist_pct.round(2),
                "Recuento": dist_cnt
            }))
    else:
        print("No hay variables categóricas con nulos.")



    # --- 3.2 Nulos en numéricas ---
    num_cols = df.select_dtypes(include=["number"]).columns
    cols_num_nulos = [c for c in num_cols if df[c].isnull().any()]

    print("\n📊 NULOS EN VARIABLES NUMÉRICAS")
    if cols_num_nulos:
        for col in cols_num_nulos:
            pct_nulos = df[col].isnull().mean() * 100
            print(f"\n📌 {col} — 🚩 FLAG: nulos_numerica ({pct_nulos:.2f}%)")

            plt.figure(figsize=(6, 4))
            s = pd.to_numeric(df[col], errors="coerce").dropna()

            # Histograma en densidad para comparar con KDE si aplica
            s.hist(bins=30, density=True, alpha=0.6)

            # KDE solo si hay suficientes datos y variabilidad
            if (len(s) >= 20) and (s.nunique() > 1) and (s.std() > 0):
                try:
                    s.plot(kind="kde", linewidth=2)
                except Exception:
                    pass

            media = s.mean()
            mediana = s.median()
            moda = s.mode()[0]

            plt.axvline(media, color="red", linestyle="--", label=f"Media: {media:.2f}")
            plt.axvline(mediana, color="green", linestyle="--", label=f"Mediana: {mediana:.2f}")
            if not np.isnan(moda):
                plt.axvline(moda, color="blue", linestyle="--", label=f"Moda: {moda:.2f}")

            plt.title(f"Distribución de {col}")
            plt.xlabel(col)
            plt.ylabel("Densidad")
            plt.legend()
            plt.show()
    else:
        print("No hay variables numéricas con nulos.")

    # ==================================================
    # 4) ESTADÍSTICAS DESCRIPTIVAS + FLAGS BÁSICAS
    # ==================================================
    # Objetivo: describir numéricas/categóricas y detectar incoherencias claras
    # (negativos y varianza 0) que suelen indicar errores o variables sin valor.
    print("\n📊 ESTADÍSTICAS DESCRIPTIVAS — NUMÉRICAS")
    if len(num_cols) > 0:
        display(df[num_cols].describe().T)

        cols_negativos = df[num_cols].columns[df[num_cols].min() < 0].tolist()
        cols_var_0 = df[num_cols].columns[df[num_cols].var() == 0].tolist()

        if cols_negativos:
            print("\n🚩 FLAG: rango - Valores negativos detectados en variables numéricas")
            for col in cols_negativos:
                min_val = df[col].min()
                print(f"  - {col} (mínimo: {min_val})")
        else:
            print("\n✅ No se detectaron valores negativos en variables numéricas.")

        if cols_var_0:
            print("\n🚩 FLAG: varianza - Variables con varianza 0 (sin variabilidad)")
            for col in cols_var_0:
                print(f"  - {col}")
        else:
            print("\n✅ No se detectaron variables con varianza 0.")
    else:
        print("No hay variables numéricas para describir.")

    print("\n🔤 ESTADÍSTICAS DESCRIPTIVAS — CATEGÓRICAS")
    if len(cols_cat) > 0:
        display(df[cols_cat].describe().T)
    else:
        print("No hay variables categóricas para describir.")

    # ==================================================
    # 4.1) MUESTRA DE CATEGORÍAS ÚNICAS (TOP N)
    # ==================================================
    # Objetivo: validar formato y consistencia de variables categóricas mostrando
    # una muestra acotada.
    print("\n📝 MUESTRA DE CATEGORÍAS (TOP 10) — VARIABLES CATEGÓRICAS")
    if len(cols_cat) > 0:
        for col in cols_cat:
            n_unique = df[col].nunique(dropna=False)
            print(f"\n📌 {col} — {n_unique} valores únicos (mostrando hasta 10)")
            valores = pd.Series(df[col].dropna().unique()).head(10)
            display(pd.DataFrame(valores, columns=["Valor (muestra)"]))
    else:
        print("No hay variables categóricas.")

    # ==================================================
    # 5) OUTLIERS (IQR)
    # ==================================================
    # Objetivo: detectar valores atípicos con IQR (1.5x) como señal para posibles
    # transformaciones, winsorización o tratamiento específico en limpieza.
    print("\n⚠️ OUTLIERS — Método IQR (1.5x)")
    if len(num_cols) > 0:
        for col in num_cols:
            s = pd.to_numeric(df[col], errors="coerce").dropna()
            if s.empty:
                continue

            q1 = s.quantile(0.25)
            q3 = s.quantile(0.75)
            iqr = q3 - q1
            if iqr == 0:
                continue

            limite_inferior = q1 - 1.5 * iqr
            limite_superior = q3 + 1.5 * iqr
            outliers = ((s < limite_inferior) | (s > limite_superior)).sum()

            if outliers > 0:
                pct_out = outliers / len(s) * 100
                print(f"🚩 FLAG: outliers - {col}: {outliers} ({pct_out:.2f}%)")

                media = s.mean()
                moda = s.mode()[0]

                plt.figure(figsize=(6, 2))
                sns.boxplot(x=s)
                plt.axvline(media, color="red", linestyle="--", label=f"Media: {media:.2f}")
                if not np.isnan(moda):
                    plt.axvline(moda, color="blue", linestyle="--", label=f"Moda: {moda:.2f}")
                plt.title(f"Boxplot de {col}")
                plt.legend()
                plt.show()
    else:
        print("No hay variables numéricas para análisis de outliers.")

    print("\n🏁 REVISIÓN EDA FINALIZADA")


In [7]:
eda(df1)

📐 VISIÓN GENERAL DEL DATASET
Vista inicial (primeras 5 filas):


,Gender,Marital Status,Occupation,Educational Qualifications,Family size,Frequently used Medium,Frequently ordered Meal category,Perference,Restaurnat Rating,Delivery Rating,No. of orders placed,Delivery Time,Order Value,Ease and convenient,Self Cooking,Health Concern,Late Delivery,Poor Hygiene,Bad past experience,More Offers and Discount,Maximum wait time,Influence of rating
Age,,,,,,,,,,,,,,,,,,,,,,
20,Female,Single,Student,Post Graduate,4,Food delivery apps,Breakfast,Non Veg foods (Lunch / Dinner),1,1,150,45,1,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,30 minutes,Yes
24,Female,Single,Student,Graduate,3,Food delivery apps,Snacks,Non Veg foods (Lunch / Dinner),1,1,32,61,3,Strongly agree,Strongly agree,Strongly agree,Agree,Strongly agree,Strongly agree,Strongly agree,30 minutes,Yes
22,Male,Single,Student,Post Graduate,3,Food delivery apps,Lunch,Non Veg foods (Lunch / Dinner),3,5,193,19,1,Strongly agree,Disagree,Neutral,Neutral,Agree,Agree,Neutral,45 minutes,Yes
22,Female,Single,Student,Graduate,6,Food delivery apps,Snacks,Veg foods (Breakfast / Lunch / Dinner),2,4,60,44,1,Agree,Agree,Strongly agree,Neutral,Agree,Disagree,Strongly agree,30 minutes,Yes
22,Male,Single,Student,Post Graduate,4,Walk-in,Lunch,Non Veg foods (Lunch / Dinner),3,1,234,24,3,Agree,Agree,Strongly agree,Strongly agree,Agree,Strongly agree,Agree,30 minutes,Yes


Dimensiones: 499 filas × 22 columnas

Información general (dtypes, nulos, memoria):
<class 'pandas.core.frame.DataFrame'>
Index: 499 entries, 20 to 35
Data columns (total 22 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Gender                             498 non-null    object
 1   Marital Status                     499 non-null    object
 2   Occupation                         498 non-null    object
 3   Educational Qualifications         499 non-null    object
 4   Family size                        499 non-null    int64 
 5   Frequently used Medium             498 non-null    object
 6   Frequently ordered Meal category   498 non-null    object
 7   Perference                         498 non-null    object
 8   Restaurnat Rating                  499 non-null    int64 
 9   Delivery Rating                    499 non-null    int64 
 10  No. of orders placed               499 non-null    int6

,Tipo,Nulos,% Nulos,Valores Únicos,% Cardinalidad
Maximum wait time,object,2,0.4,5,1.00
Gender,object,1,0.2,2,0.40
Frequently used Medium,object,1,0.2,4,0.80
Ease and convenient,object,1,0.2,5,1.00
Frequently ordered Meal category,object,1,0.2,4,0.80
Occupation,object,1,0.2,4,0.80
More Offers and Discount,object,1,0.2,5,1.00
Bad past experience,object,1,0.2,5,1.00
Health Concern,object,1,0.2,5,1.00
Perference,object,1,0.2,4,0.80



🚩 FLAG: nulos - Clasificación por umbral (% nulos)
  🟢 Bajo (≤ 5%):
    - Maximum wait time (0.40%)
    - Gender (0.20%)
    - Frequently used Medium (0.20%)
    - Ease and convenient (0.20%)
    - Frequently ordered Meal category  (0.20%)
    - Occupation (0.20%)
    - More Offers and Discount (0.20%)
    - Bad past experience (0.20%)
    - Health Concern (0.20%)
    - Perference (0.20%)
    - Delivery Time (0.20%)
  🟡 Moderado (5%–20%):
    - Ninguna
  🔴 Alto (> 20%):
    - Ninguna

🔤 NULOS EN VARIABLES CATEGÓRICAS

📌 Gender — 🚩 FLAG: nulos_categorica (0.20%)


...


📌 Maximum wait time — 6 valores únicos (mostrando hasta 10)


,Valor (muestra)
0,30 minutes
1,45 minutes
2,60 minutes
3,More than 60 minutes
4,15 minutes



📌 Influence of rating — 3 valores únicos (mostrando hasta 10)


,Valor (muestra)
0,Yes
1,Maybe
2,No



⚠️ OUTLIERS — Método IQR (1.5x)

🏁 REVISIÓN EDA FINALIZADA


In [10]:
eda(df2)

📐 VISIÓN GENERAL DEL DATASET
Vista inicial (primeras 5 filas):


,restaurant_id,food_item,order_time,delivery_time,delivery_distance,order_value,delivery_method,traffic_condition,weather_condition,delivery_delay,route_taken,customer_id,age,gender,location,order_history,customer_rating,preferred_cuisine,order_frequency,loyalty_program,food_temperature,food_freshness,packaging_quality,food_condition,customer_satisfaction,small_route,bike_friendly_route,route_type,route_efficiency,traffic_avoidance
order_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ORD000001,16,Taccos,2024-01-31,2024-01-31,2.17,42.21,Bike,Medium,Snowy,17.15,Route_4,CUST000001,32,Female,Hyderabad,46,4,American,Monthly,Yes,Hot,5,1,Fair,3,False,False,Bike-friendly,0.801291,Yes
ORD000002,30,Briyani rice,2024-10-16,2024-10-16,13.40,24.82,Car,High,Sunny,16.04,Route_2,CUST000002,59,Male,Lucknow,19,1,Asian,Weekly,No,Warm,3,2,Fair,5,False,False,Bike-friendly,0.645795,No
ORD000003,3,Pasta,2024-09-11,2024-09-11,10.74,37.25,Walk,High,Snowy,9.09,Route_3,CUST000003,20,Male,Chennai,10,5,Mexican,Monthly,Yes,Warm,4,5,Good,5,True,False,Bike-friendly,0.291193,No
ORD000004,93,Whole cake,2024-09-17,2024-09-17,6.29,49.88,Bike,High,Snowy,-2.11,Route_5,CUST000004,31,Male,Bangalore,6,4,American,Monthly,Yes,Cold,2,3,Poor,4,False,False,Car-only,0.133827,No
ORD000005,15,Sushi,2024-08-26,2024-08-26,2.94,8.53,Walk,Low,Sunny,12.20,Route_5,CUST000005,27,Other,Ahmedabad,24,3,Asian,Weekly,Yes,Cold,4,5,Good,1,False,True,Car-only,0.349233,No


Dimensiones: 20000 filas × 30 columnas

Información general (dtypes, nulos, memoria):
<class 'pandas.core.frame.DataFrame'>
Index: 20000 entries, ORD000001 to ORD020000
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   restaurant_id          20000 non-null  int64  
 1   food_item              20000 non-null  object 
 2   order_time             20000 non-null  object 
 3   delivery_time          20000 non-null  object 
 4   delivery_distance      20000 non-null  float64
 5   order_value            20000 non-null  float64
 6   delivery_method        20000 non-null  object 
 7   traffic_condition      20000 non-null  object 
 8   weather_condition      20000 non-null  object 
 9   delivery_delay         20000 non-null  float64
 10  route_taken            20000 non-null  object 
 11  customer_id            20000 non-null  object 
 12  age                    20000 non-null  int64  
 13  gender       

,Tipo,Nulos,% Nulos,Valores Únicos,% Cardinalidad
restaurant_id,int64,0,0.0,100,0.50
food_item,object,0,0.0,21,0.10
order_time,object,0,0.0,347,1.74
delivery_time,object,0,0.0,347,1.74
delivery_distance,float64,0,0.0,1301,6.50
order_value,float64,0,0.0,4026,20.13
delivery_method,object,0,0.0,3,0.02
traffic_condition,object,0,0.0,3,0.02
weather_condition,object,0,0.0,3,0.02
delivery_delay,float64,0,0.0,2905,14.52



🚩 FLAG: nulos - Clasificación por umbral (% nulos)
  🟢 Bajo (≤ 5%):
    - Ninguna
  🟡 Moderado (5%–20%):
    - Ninguna
  🔴 Alto (> 20%):
    - Ninguna

🔤 NULOS EN VARIABLES CATEGÓRICAS
No hay variables categóricas con nulos.

📊 NULOS EN VARIABLES NUMÉRICAS
No hay variables numéricas con nulos.

📊 ESTADÍSTICAS DESCRIPTIVAS — NUMÉRICAS


...


📌 route_type — 3 valores únicos (mostrando hasta 10)


,Valor (muestra)
0,Bike-friendly
1,Car-only
2,Mixed



📌 traffic_avoidance — 2 valores únicos (mostrando hasta 10)


,Valor (muestra)
0,Yes
1,No



⚠️ OUTLIERS — Método IQR (1.5x)

🏁 REVISIÓN EDA FINALIZADA
